# Sensibilizar el punto exacto: de "estación cercana" a "clima en el punto del cliente" (ECO | Wind)

Hallazgo 24 resolvió cómo encontrar y descargar la estación REAL más cercana a cualquier punto, en
cualquiera de los 20 países del catálogo -- pero "más cercana" no es "exacta" (Bogotá salió a 84.6
km de su estación más cercana). Este notebook prueba el paso que falta: ajustar la MAGNITUD de la
forma real de la estación donante a lo que pasa en el punto EXACTO que pide el cliente, sin volver
a anclar nada a San José ni a ningún sitio fijo.

**Mecanismo propuesto (ver el plan completo en el chat):**

```
media_ajustada_al_punto_exacto = media_real_de_la_estación_donante × [fuente_continua(punto_exacto) / fuente_continua(ubicación_de_la_estación)]
```

La `fuente_continua` sólo se usa para una RAZÓN entre dos puntos cercanos, no para su valor
absoluto -- si esa fuente tiene un sesgo sistemático en la región (NASA POWER subestima ~3x en
Costa Rica, Hallazgo 1), ese sesgo se cancela en gran parte al dividir, y lo que sobrevive es la
diferencia real de microclima entre la estación y el punto exacto. La FORMA horaria (variabilidad,
patrón diurno/estacional) sigue siendo 100% real, de la estación donante -- no se inventa nada,
sólo se reescala la magnitud.

Dos partes: (1) investigar en vivo si el Global Wind Atlas tiene una API de consulta por punto real
y accesible -- no se encontró un endpoint documentado y confirmado en la investigación previa (sólo
"la API existe, no está pensada para bulk"), así que esta parte SÓLO prueba alcanzabilidad de las
páginas reales encontradas, no inventa una URL de datos; (2) el mecanismo completo usando NASA
POWER como fuente continua -- ya confirmado que funciona en Colab (Hallazgo 23), es la vía segura
mientras (1) se termina de confirmar.

In [1]:
import os

def _find_repo_root():
    for candidato in ("..", "/content/ECO-Wind"):
        if os.path.exists(os.path.join(candidato, ".git")):
            return os.path.abspath(candidato)
    return None

repo = _find_repo_root()
if repo is None:
    repo = "/content/ECO-Wind"
    get_ipython().system(f"git clone https://github.com/Sogo2012/ECO-Wind.git {repo}")
else:
    get_ipython().system(f"git -C {repo} fetch origin main")
    get_ipython().system(f"git -C {repo} reset --hard origin/main")

get_ipython().run_line_magic("cd", f"{repo}/notebooks")
get_ipython().system(f"git -C {repo} log -1 --format='Commit activo: %h  %s  (%ci)'")

From https://github.com/Sogo2012/eco-wind
 * branch            main       -> FETCH_HEAD


HEAD is now at dd824f5 docs(fase2): documentar Hallazgo 25 en avance-de-proyecto.md


/home/user/eco-wind/notebooks
Commit activo: dd824f5  docs(fase2): documentar Hallazgo 25 en avance-de-proyecto.md  (2026-08-31 22:03:57 +0000)


In [2]:
import sys
sys.path.insert(0, "..")

import calendar

import numpy as np
import pandas as pd
import requests

from engine.formas_regionales import cargar_formas_conocidas, vecino_mas_cercano
from engine.simulador_pista_a import generar_clima_gwa, simular

pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

## Parte 1 — ¿Tiene el Global Wind Atlas una API de punto real y accesible?

**Honesto de entrada:** la investigación previa (WebSearch) confirmó que GWA 4.0 tiene cobertura
GLOBAL (todos los países + zonas offshore) y que existe un "EMD-API - Global Atlas Services"
documentado como REST/OpenAPI -- pero NO se encontró un endpoint concreto y confirmado para
consulta por punto (lat/lon → media de viento en JSON). `help.emd.dk` (donde vive esa
documentación) ya está confirmado bloqueado en el sandbox de desarrollo (Hallazgo 2) -- acá se
prueba si sigue bloqueado desde Colab, y se revisa el contenido de las páginas reales que sí se
encontraron, para ver si documentan el formato del endpoint. No se inventa ninguna URL de datos.

In [3]:
paginas_reales_a_probar = {
    "Global Wind Atlas (home)": "https://globalwindatlas.info",
    "GWA -- GIS files & API access": "https://globalwindatlas.info/download/gis-files",
    "EMD-API docs (Wiki-WindPRO)": "https://help.emd.dk/mediawiki/index.php/EMD-API_-_Global_Atlas_Services",
    "windatlas.xyz docs (tool de terceros, no es GWA/DTU)": "http://windatlas.xyz/docs/api/",
}

for nombre, url in paginas_reales_a_probar.items():
    try:
        resp = requests.get(url, timeout=15)
        print(f"{nombre}: OK -- {resp.status_code}, {len(resp.text)} caracteres")
        if resp.status_code == 200 and ("api" in resp.text.lower() or "endpoint" in resp.text.lower()):
            print("  (la página menciona 'api'/'endpoint' -- vale la pena leerla completa a mano)")
    except Exception as exc:
        print(f"{nombre}: FALLO -- {exc!r}")

Global Wind Atlas (home): FALLO -- ProxyError(MaxRetryError("HTTPSConnectionPool(host='globalwindatlas.info', port=443): Max retries exceeded with url: / (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))


GWA -- GIS files & API access: FALLO -- ProxyError(MaxRetryError("HTTPSConnectionPool(host='globalwindatlas.info', port=443): Max retries exceeded with url: /download/gis-files (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))


EMD-API docs (Wiki-WindPRO): FALLO -- ProxyError(MaxRetryError("HTTPSConnectionPool(host='help.emd.dk', port=443): Max retries exceeded with url: /mediawiki/index.php/EMD-API_-_Global_Atlas_Services (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))
windatlas.xyz docs (tool de terceros, no es GWA/DTU): OK -- 403, 100 caracteres


**Actualización real (Colab, 31 de agosto 2026, confirmado por Pablo con captura de pantalla):**
las 4 páginas SÍ responden 200 en Colab (bloqueadas sólo en este sandbox de desarrollo). La página
real de "GIS files & API access" (`globalwindatlas.info/download/gis-files`) contesta la pregunta
directamente -- **no hay una API de consulta por punto separada.** El formulario real es: elegir
país → elegir capa (GEOJSON, WIND-SPEED, POWER-DENSITY, AIR-DENSITY, ...) → elegir altura (10, 50,
100, 150, 200m) → esto genera una URL de descarga de RASTER PARA TODO EL PAÍS, y esa MISMA URL
"can also be used as an API service" -- es decir, el "API" de GWA es exactamente el mecanismo que
ya se había construido para Costa Rica en Hallazgo 17 (`descargar_raster_costa_rica()`), sólo que
generalizable a cualquier país. La página también advierte explícito: **"not to be used for bulk
downloads of all countries or datasets"** -- bajar UN país está bien (como ya se hace), scriptear
una descarga masiva de los 20 no.

El patrón de URL ya estaba confirmado desde Hallazgo 17 y coincide exacto con lo que muestra la
página real: `https://globalwindatlas.info/api/gis/country/{ISO3}/{capa}/{altura}`. Generalizado
ahora en `engine/gwa_raster.py::descargar_raster_pais(pais_iso3, altura=10)` -- ver Parte 3 más
abajo, donde se usa exactamente esto (con Costa Rica) en vez de NASA POWER para el ajuste espacial.

## Parte 2 — El mecanismo completo, con NASA POWER como fuente continua (la vía ya confirmada)

`factor_ajuste_nasa_power()`: la razón entre la media de NASA POWER en el punto exacto y en la
ubicación de la estación donante. `evaluar_punto_con_ajuste()`: junta todo -- encuentra el vecino
real más cercano (reusa `vecino_mas_cercano()` de `engine/formas_regionales.py`, Hallazgo 21), y en
vez de escalar su forma a una media "ya conocida" (que en un punto nuevo de verdad NUNCA se tiene),
la escala por este factor de ajuste espacial -- es una prueba más honesta que la de Hallazgo 21/22,
porque no usa ninguna información que no estaría disponible para un punto nuevo real.

In [4]:
NASA_POWER_HOURLY_URL = "https://power.larc.nasa.gov/api/temporal/hourly/point"


def fetch_nasa_power_hourly(lat, lon, year, community="SB", parameters=("WS10M",)):
    params = {
        "parameters": ",".join(parameters), "community": community,
        "longitude": lon, "latitude": lat,
        "start": f"{year}0101", "end": f"{year}1231", "format": "JSON",
    }
    resp = requests.get(NASA_POWER_HOURLY_URL, params=params, timeout=60)
    resp.raise_for_status()
    df = pd.DataFrame(resp.json()["properties"]["parameter"])
    df.index = pd.to_datetime(df.index, format="%Y%m%d%H")
    horas_esperadas = 8784 if calendar.isleap(year) else 8760
    if len(df) != horas_esperadas:
        raise ValueError(f"Esperaba {horas_esperadas} horas, llegaron {len(df)}")
    return df


def factor_ajuste_nasa_power(lat_exacto, lon_exacto, lat_estacion, lon_estacion, year=2023):
    '''
    Razon NASA POWER(punto exacto) / NASA POWER(ubicacion de la estacion donante) -- el sesgo
    sistematico de NASA POWER (Hallazgo 1) se cancela en gran parte al dividir dos puntos
    cercanos de la misma fuente; sobrevive sobre todo la diferencia real de microclima.
    '''
    media_exacto = fetch_nasa_power_hourly(lat_exacto, lon_exacto, year)["WS10M"].mean()
    media_estacion = fetch_nasa_power_hourly(lat_estacion, lon_estacion, year)["WS10M"].mean()
    return media_exacto / media_estacion, media_exacto, media_estacion


def evaluar_punto_con_ajuste(lat, lon, formas, excluir=None, year=2023,
                              modelo="medium_tulip", N=3, altura_buje=3.0, elevacion_m=0.0):
    clave_donante, dist_km = vecino_mas_cercano(lat, lon, formas, excluir=excluir)
    donante = formas[clave_donante]

    factor, media_np_exacto, media_np_donante = factor_ajuste_nasa_power(
        lat, lon, donante["lat"], donante["lon"], year=year)

    media_donante_real = (float(np.mean([r["val"] for r in donante["ws_json"]])) if clave_donante == "san_jose"
                           else float(donante["df_real"]["WS10M"].mean()))
    media_ajustada = media_donante_real * factor

    df_clima, _ = generar_clima_gwa(donante["ws_json"], donante["hm_json"], media_objetivo=media_ajustada)
    r = simular(df_clima, altura_buje, modelo, N, elevacion_m=elevacion_m)

    return dict(donante=formas[clave_donante]["nombre"], distancia_km=dist_km, factor_ajuste=factor,
                media_np_exacto=media_np_exacto, media_np_donante=media_np_donante,
                media_donante_real=media_donante_real, media_ajustada=media_ajustada,
                kwh_ajustado=r["kwh_anual"])

## Validación leave-one-out con el mecanismo nuevo -- ¿mejora sobre lo ya documentado?

Para cada uno de los 4 sitios reales conocidos: se tapa su propia forma Y su propia media real (a
diferencia de Hallazgo 21/22, acá NO se usa la media real ya conocida del sitio -- es información
que un punto nuevo de verdad no tendría). Se compara contra la verdad real ya conocida, y contra
los dos mecanismos ya documentados (siempre San José, y vecino más cercano con curva por residuo
de Hallazgo 22).

In [5]:
formas = cargar_formas_conocidas(usar_residuo=True)
filas = []

for clave, sitio in formas.items():
    if clave == "san_jose":
        df_real, _ = generar_clima_gwa(sitio["ws_json"], sitio["hm_json"])
        media_real = float(np.mean([r["val"] for r in sitio["ws_json"]]))
    else:
        df_real = sitio["df_real"]
        media_real = float(df_real["WS10M"].mean())
    r_real = simular(df_real, 3.0, "medium_tulip", 3, elevacion_m=sitio["elevacion_m"])

    try:
        ajuste = evaluar_punto_con_ajuste(sitio["lat"], sitio["lon"], formas, excluir=clave,
                                           elevacion_m=sitio["elevacion_m"])
        error_ajustado_pct = (ajuste["kwh_ajustado"] / r_real["kwh_anual"] - 1) * 100
        fila = dict(sitio=sitio["nombre"], kwh_real=r_real["kwh_anual"],
                    donante=ajuste["donante"], distancia_km=ajuste["distancia_km"],
                    factor_ajuste_nasa_power=ajuste["factor_ajuste"],
                    kwh_nuevo_ajustado=ajuste["kwh_ajustado"], error_nuevo_ajustado_pct=error_ajustado_pct)
    except Exception as exc:
        fila = dict(sitio=sitio["nombre"], kwh_real=r_real["kwh_anual"],
                    donante=None, distancia_km=None, factor_ajuste_nasa_power=None,
                    kwh_nuevo_ajustado=None, error_nuevo_ajustado_pct=f"FALLO: {exc!r}")
    filas.append(fila)

pd.DataFrame(filas)

sitio	kwh_real	donante	distancia_km	factor_ajuste_nasa_power	kwh_nuevo_ajustado	error_nuevo_ajustado_pct
San José (Aeropuerto Juan Santamaría)	156.439	Nicoya A.P. (Guanacaste, Pacífico seco)	137.458	0.365	2.303	-98.528
Nicoya A.P. (Guanacaste, Pacífico seco)	52.400	Daniel Oduber / Liberia Intl. A.P. (Guanacaste)	50.321	0.963	363.326	593.374
Daniel Oduber / Liberia Intl. A.P. (Guanacaste)	291.487	Nicoya A.P. (Guanacaste, Pacífico seco)	50.321	1.038	71.785	-75.373
Finca Favorita (Limón, Caribe)	7.439	San José (Aeropuerto Juan Santamaría)	178.604	1.880	1136.978	15183.964


## Diagnóstico honesto del resultado de NASA POWER -- no es un bug, es el método fallando de raíz

Números catastróficos: San José -98.5%, Nicoya +593%, Liberia -75.4%, **Finca Favorita +15,184%**
(153x la producción real). Verificado con cálculo, no es un error de fórmula:

- **San José vs. Finca Favorita:** NASA POWER dice que Finca Favorita es 1.88x más ventosa que San
  José. La VERDAD es lo contrario -- Finca Favorita tiene sólo 38.5% del viento de San José. La
  razón sale literalmente al revés.
- **Nicoya vs. Liberia** (50km, terreno similar): en la realidad Liberia es 1.74x más ventosa. NASA
  POWER casi no distingue los dos puntos (factor 0.96-1.04) -- su grilla de ~50-60km es demasiado
  gruesa incluso para dos sitios del mismo tipo de terreno.

La idea de que "la razón entre dos puntos cercanos cancela el sesgo sistemático de la fuente" ASUME
que el sesgo es parejo en la región -- pero Hallazgo 1 ya había mostrado que el sesgo de NASA POWER
viene de no poder resolver terreno complejo (por eso subestima ~3x en el valle de San José). Si el
sesgo depende de qué tan complejo es el terreno de CADA punto, la razón no cancela nada -- puede
invertir la relación real, que es exactamente lo que pasó. **Conclusión: para este terreno, NASA
POWER no sirve como corrector espacial -- no es un tema de afinar el método, hay que cambiar de
fuente.**

## Parte 3 — El mismo mecanismo, con GWA (250m) en vez de NASA POWER (~50-60km)

Si el problema de NASA POWER es resolución (no puede ver el terreno complejo de Costa Rica), GWA
debería andar mucho mejor -- es ~200-1000x más fino. Mecánicamente es más simple además: en vez de
2 llamadas a una API por internet, son 2 lecturas de píxel del MISMO raster ya descargado una vez
(`factor_ajuste_gwa()`, nuevo en `engine/gwa_raster.py`, mismo patrón que
`factor_ajuste_nasa_power()` de la Parte 2).

**Necesita el ráster de Costa Rica descargado** (`descargar_raster_pais("CRI")`, generalización de
`descargar_raster_costa_rica()` de Hallazgo 17) -- no corre en este sandbox, sólo en Colab.

In [6]:
from engine.gwa_raster import descargar_raster_pais, factor_ajuste_gwa, RUTA_RASTER_CR_DEFAULT

try:
    ruta_raster_cr = descargar_raster_pais("CRI", altura=10)
    print(f"Ráster descargado: {ruta_raster_cr}")
except Exception as exc:
    ruta_raster_cr = RUTA_RASTER_CR_DEFAULT
    print(f"No se pudo descargar (mismo bloqueo de red ya documentado, Hallazgo 2): {exc!r}")
    print("La celda de abajo va a fallar igual (no hay archivo local) hasta correr esto en Colab.")


No se pudo descargar (mismo bloqueo de red ya documentado, Hallazgo 2): ProxyError(MaxRetryError("HTTPSConnectionPool(host='globalwindatlas.info', port=443): Max retries exceeded with url: /api/gis/country/CRI/wind-speed/10 (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))
La celda de abajo va a fallar igual (no hay archivo local) hasta correr esto en Colab.


### Diagnóstico previo -- ¿el RÁSTER CRUDO de GWA ya se acerca a la realidad en los 4 sitios?

Antes de ver si el MECANISMO de razón funciona, un chequeo más simple y directo: leer el valor
crudo del ráster (sin ninguna razón, sin ningún ajuste) en la coordenada exacta de cada uno de los
4 sitios reales, y compararlo contra su media real ya conocida. Esto es justo lo que faltó
diagnosticar ANTES de correr NASA POWER (Hallazgo 25) -- ahora se hace primero, para no necesitar
otro viaje a Colab si el resultado de la razón sale raro otra vez.

In [7]:
from engine.gwa_raster import muestrear_velocidad_media

print("Valor crudo del ráster GWA (10m) vs. media real conocida, en la propia coordenada de cada sitio:")
filas_diag_gwa = []
for clave, sitio in formas.items():
    media_real = (float(np.mean([r["val"] for r in sitio["ws_json"]])) if clave == "san_jose"
                  else float(sitio["df_real"]["WS10M"].mean()))
    try:
        media_gwa = muestrear_velocidad_media(sitio["lat"], sitio["lon"], ruta_raster_cr)
        diff_pct = (media_gwa / media_real - 1) * 100
    except Exception as exc:
        media_gwa, diff_pct = None, f"FALLO: {exc!r}"
    filas_diag_gwa.append(dict(sitio=sitio["nombre"], media_real_m_s=media_real,
                                media_gwa_raster=media_gwa, diferencia_pct=diff_pct))

pd.DataFrame(filas_diag_gwa)

Valor crudo del ráster GWA (10m) vs. media real conocida, en la propia coordenada de cada sitio:


,sitio,media_real_m_s,media_gwa_raster,diferencia_pct
0,San José (Aeropuerto Juan Santamaría),3.669,None,FALLO: FileNotFoundError('No existe /home/user...
1,"Nicoya A.P. (Guanacaste, Pacífico seco)",2.091,None,FALLO: FileNotFoundError('No existe /home/user...
2,Daniel Oduber / Liberia Intl. A.P. (Guanacaste...,3.629,None,FALLO: FileNotFoundError('No existe /home/user...
3,"Finca Favorita (Limón, Caribe)",1.413,None,FALLO: FileNotFoundError('No existe /home/user...


In [8]:
def evaluar_punto_con_ajuste_gwa(lat, lon, formas, excluir=None, ruta_raster=RUTA_RASTER_CR_DEFAULT,
                                  modelo="medium_tulip", N=3, altura_buje=3.0, elevacion_m=0.0):
    clave_donante, dist_km = vecino_mas_cercano(lat, lon, formas, excluir=excluir)
    donante = formas[clave_donante]

    factor, media_gwa_exacto, media_gwa_donante = factor_ajuste_gwa(
        lat, lon, donante["lat"], donante["lon"], ruta_raster=ruta_raster)

    media_donante_real = (float(np.mean([r["val"] for r in donante["ws_json"]])) if clave_donante == "san_jose"
                           else float(donante["df_real"]["WS10M"].mean()))
    media_ajustada = media_donante_real * factor

    df_clima, _ = generar_clima_gwa(donante["ws_json"], donante["hm_json"], media_objetivo=media_ajustada)
    r = simular(df_clima, altura_buje, modelo, N, elevacion_m=elevacion_m)

    return dict(donante=formas[clave_donante]["nombre"], distancia_km=dist_km, factor_ajuste=factor,
                media_gwa_exacto=media_gwa_exacto, media_gwa_donante=media_gwa_donante,
                media_donante_real=media_donante_real, media_ajustada=media_ajustada,
                kwh_ajustado=r["kwh_anual"])


filas_gwa = []
for clave, sitio in formas.items():
    if clave == "san_jose":
        df_real, _ = generar_clima_gwa(sitio["ws_json"], sitio["hm_json"])
        media_real = float(np.mean([r["val"] for r in sitio["ws_json"]]))
    else:
        df_real = sitio["df_real"]
        media_real = float(df_real["WS10M"].mean())
    r_real = simular(df_real, 3.0, "medium_tulip", 3, elevacion_m=sitio["elevacion_m"])

    try:
        ajuste = evaluar_punto_con_ajuste_gwa(sitio["lat"], sitio["lon"], formas, excluir=clave,
                                               elevacion_m=sitio["elevacion_m"])
        error_pct = (ajuste["kwh_ajustado"] / r_real["kwh_anual"] - 1) * 100
        fila = dict(sitio=sitio["nombre"], kwh_real=r_real["kwh_anual"], donante=ajuste["donante"],
                    distancia_km=ajuste["distancia_km"], factor_ajuste_gwa=ajuste["factor_ajuste"],
                    kwh_nuevo_gwa=ajuste["kwh_ajustado"], error_nuevo_gwa_pct=error_pct)
    except Exception as exc:
        fila = dict(sitio=sitio["nombre"], kwh_real=r_real["kwh_anual"], donante=None,
                    distancia_km=None, factor_ajuste_gwa=None, kwh_nuevo_gwa=None,
                    error_nuevo_gwa_pct=f"FALLO: {exc!r}")
    filas_gwa.append(fila)

pd.DataFrame(filas_gwa)

,sitio,kwh_real,donante,distancia_km,factor_ajuste_gwa,kwh_nuevo_gwa,error_nuevo_gwa_pct
0,San José (Aeropuerto Juan Santamaría),156.439,None,None,None,None,FALLO: FileNotFoundError('No existe /home/user...
1,"Nicoya A.P. (Guanacaste, Pacífico seco)",52.400,None,None,None,None,FALLO: FileNotFoundError('No existe /home/user...
2,Daniel Oduber / Liberia Intl. A.P. (Guanacaste...,291.487,None,None,None,None,FALLO: FileNotFoundError('No existe /home/user...
3,"Finca Favorita (Limón, Caribe)",7.439,None,None,None,None,FALLO: FileNotFoundError('No existe /home/user...


## Conclusión final -- comparar las 4 vías, con números reales donde ya los hay

| Sitio | Siempre San José (H21) | Vecino+residuo, con verdad conocida (H22) | Vecino+NASA POWER, sin atajos (H25) | Vecino+GWA, sin atajos (H25) |
|---|---|---|---|---|
| San José | — | — | **-98.5%** | *(ver tabla de arriba)* |
| Nicoya | -41.5% | +47.6% | **+593.4%** | *(ver tabla de arriba)* |
| Liberia | -43.7% | +15.9% | **-75.4%** | *(ver tabla de arriba)* |
| Finca Favorita | +19.2% | +19.2% | **+15,184.0%** | *(ver tabla de arriba)* |

NASA POWER como corrector espacial queda **descartado** para este terreno -- no es cuestión de
afinar el método, su resolución (~50-60km) no puede ver la diferencia de microclima entre puntos
que en la realidad son muy distintos. Si la columna de GWA sale razonable (error de una o dos
cifras, no miles de por ciento), es la señal de que el camino correcto es GWA -- más fino, y ahora
confirmado que se puede automatizar (país por país, no en bulk) exactamente igual que ya se hace
con Costa Rica. Si GWA también falla, hay que reconsiderar el mecanismo de ajuste espacial en sí
(quizás una fuente todavía más fina, como ERA5 a nivel local, o directamente confiar más en la
estación real más cercana sin ningún ajuste cuando está razonablemente cerca).

No conectado a `app.py` todavía -- sigue siendo investigación.